# FingerBall Stage 2F — Interactive 3D Hand Skeleton Viewer

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_v2_Stage2F_3D_Hand_Skeleton_Viewer.ipynb)

MediaPipe Hand Landmarkerの21点2D＋相対Zを利用した**時間スライダー＋縦横360°回転**ビューア。**デフォルトの原点はカメラ画像左上 (0,0)** です。手首の画面内移動を固定座標軸上で可視化し、手首の軌跡も描画します。

**注意：この座標系はカメラ画像座標系であり、実空間のカメラ3D座標系ではありません。X/Yは元動画の座標、Zは手首基準の推定相対深度です。** 実寸3Dや測定された360度映像ではありません。

In [ ]:
%pip -q install "plotly>=5.18" numpy pandas matplotlib opencv-python-headless tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json,zipfile,math
import numpy as np,pandas as pd,plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pyplot as plt,cv2
from tqdm.auto import tqdm

## 1. Google DriveのCSVを選択・設定

In [ ]:
PROJECT_PATH='/content/drive/MyDrive/母指回内トラッキング'
SOURCE_STAGE='stage2d' # 'stage2d' または 'stage2e'
CSV_PATH='' # 空なら指定Stageから自動選択
VIDEO_STEM='7A5812501DC5F7120DF842D06EE1B74C695459F1'
START_FRAME,END_FRAME=175,210
REPRESENTATIVE_FRAME=198
Z_DEPTH_MULTIPLIER=1.0
COORDINATE_MODE='image'  # 'image': 画像左上原点（推奨）、'wrist': 手首原点
IMAGE_HEIGHT_PX=648  # 動画の高さ。y_normがCSVにあれば自動推定
SHOW_WRIST_TRAJECTORY=True
ROTATION_FRAMES=72
ROTATION_FPS=24
EXPORT_MP4=True
project=Path(PROJECT_PATH)
if CSV_PATH: csv_path=Path(CSV_PATH)
else:
    root=project/f'FingerBall_v2_{SOURCE_STAGE}_results'
    csv_path=root/VIDEO_STEM/'hand_21_landmarks_long.csv'
    if not csv_path.exists():
        choices=sorted(root.glob('*/hand_21_landmarks_long.csv'),key=lambda p:p.stat().st_mtime,reverse=True)
        if not choices:raise FileNotFoundError('CSVが見つかりません。CSV_PATHを指定してください')
        csv_path=choices[0]
if not csv_path.is_file():raise FileNotFoundError(csv_path)
out_dir=project/'FingerBall_v2_stage2f_results'/csv_path.parent.name
out_dir.mkdir(parents=True,exist_ok=True)
print('CSV:',csv_path,'\nOutput:',out_dir)

## 2. 座標系：画像左上を原点とする疑似3D（X/Yは画面座標、Zは手首相対）

**画像原点モード**：X=x_px/W、Y=y_px/W（下向きが正）、Z=−(z_relative−z_wrist)×係数。X/Yは手首を引かないため、手首の画像内位置が保持されます。画像の縦横比をX/Y軸に反映します。

**手首原点モード**：従来どおりX=(x−x_wrist)/W、Y=−(y−y_wrist)/W。Zはどちらでも手首相対のままです。

実際のカメラからの距離（絶対奥行き）は不明です。

In [ ]:
df=pd.read_csv(csv_path)
req={'frame','landmark_id','x_px','y_px','z_relative','detected'}
if req-set(df.columns):raise ValueError('CSVの必須列がありません: '+str(req-set(df.columns)))
if df.duplicated(['frame','landmark_id']).any():raise ValueError('frame/landmark_idに重複があります')
for key in ['x_px','y_px','z_relative']:df[key]=pd.to_numeric(df[key],errors='coerce')
WIDTH=1206.
if 'x_norm' in df.columns:
    a=df[['x_px','x_norm']].dropna()
    a=a[a.x_norm.abs()>1e-7]
    if len(a):WIDTH=float(np.median(a.x_px/a.x_norm))
HEIGHT=float(IMAGE_HEIGHT_PX)
if 'y_norm' in df.columns:
    b=df[['y_px','y_norm']].dropna()
    b=b[b.y_norm.abs()>1e-7]
    if len(b):HEIGHT=float(np.median(b.y_px/b.y_norm))
if not np.isfinite(WIDTH) or WIDTH<=0 or not np.isfinite(HEIGHT) or HEIGHT<=0:
    raise ValueError('画像の幅/高さを確認してください')
if COORDINATE_MODE not in ('image','wrist'):
    raise ValueError("COORDINATE_MODEは'image'または'wrist'にしてください")
FINGERS={
 'Thumb':[(0,1),(1,2),(2,3),(3,4)],
 'Index':[(0,5),(5,6),(6,7),(7,8)],
 'Middle':[(0,9),(9,10),(10,11),(11,12)],
 'Ring':[(0,13),(13,14),(14,15),(15,16)],
 'Little':[(0,17),(17,18),(18,19),(19,20)]}
COLORS={'Thumb':'#f59e0b','Index':'#06b6d4','Middle':'#d946ef','Ring':'#94a3b8','Little':'#64748b'}
TIPS={'Thumb':4,'Index':8,'Middle':12}
frames=list(range(START_FRAME,END_FRAME+1));coords={}
for f in frames:
    g=df[df.frame.eq(f)&df.landmark_id.between(0,20)].sort_values('landmark_id')
    if len(g)!=21 or g.landmark_id.tolist()!=list(range(21)):
        coords[f]=None;continue
    xyz=g[['x_px','y_px','z_relative']].to_numpy(float)
    if not np.isfinite(xyz).all() or not np.all(g.detected.astype(int).to_numpy()==1):
        coords[f]=None;continue
    if COORDINATE_MODE=='image':
        coords[f]=np.column_stack([xyz[:,0]/WIDTH,xyz[:,1]/WIDTH,
              -(xyz[:,2]-xyz[0,2])*Z_DEPTH_MULTIPLIER])
    else:
        relative=xyz-xyz[0]
        coords[f]=np.column_stack([relative[:,0]/WIDTH,-relative[:,1]/WIDTH,
              -relative[:,2]*Z_DEPTH_MULTIPLIER])
valid=[f for f in frames if coords[f] is not None]
if not valid:raise RuntimeError('有効な21点フレームがありません')
if REPRESENTATIVE_FRAME not in valid:
    old=REPRESENTATIVE_FRAME;REPRESENTATIVE_FRAME=min(valid,key=lambda f:abs(f-old))
    print('代表フレームを変更:',old,'->',REPRESENTATIVE_FRAME)
ZR=max(.05,max(float(np.max(np.abs(coords[f][:,2]))) for f in valid)*1.5)
R=max(.06,max(float(np.max(np.abs(coords[f]))) for f in valid)*1.2)
if COORDINATE_MODE=='image':
    X_RANGE=[-.04,1.04]
    Y_RANGE=[-.04,HEIGHT/WIDTH+.04]
    Z_RANGE=[-ZR,ZR]
    ASPECT=dict(x=1,y=HEIGHT/WIDTH,z=.5)
else:
    X_RANGE=Y_RANGE=Z_RANGE=[-R,R]
    ASPECT=dict(x=1,y=1,z=1)
print('Valid:',len(valid),'/',len(frames),
      'representative:',REPRESENTATIVE_FRAME,'mode:',COORDINATE_MODE,
      'image size:',round(WIDTH),round(HEIGHT))

## 3. 時間スライダー・マウス360度回転付きPlotlyビューア

In [ ]:
def traces(p):
    result=[]
    for finger,edges in FINGERS.items():
        x=[];y=[];z=[]
        for a,b in edges:
            for k in (a,b):
                if p is None:x.append(None);y.append(None);z.append(None)
                else:x.append(float(p[k,0]));y.append(float(p[k,1]));z.append(float(p[k,2]))
            x.append(None);y.append(None);z.append(None)
        result.append(go.Scatter3d(x=x,y=y,z=z,mode='lines',
            line=dict(color=COLORS[finger],width=9),name=finger))
    for finger,j in TIPS.items():
        v=None if p is None else p[j]
        result.append(go.Scatter3d(x=[] if v is None else [float(v[0])],
            y=[] if v is None else [float(v[1])],z=[] if v is None else [float(v[2])],
            mode='markers',marker=dict(color=COLORS[finger],size=7),name=finger+' tip',showlegend=False))
    wrist=None if p is None else p[0]
    result.append(go.Scatter3d(
        x=[] if wrist is None else [float(wrist[0])],
        y=[] if wrist is None else [float(wrist[1])],
        z=[] if wrist is None else [float(wrist[2])],
        mode='markers',marker=dict(color='#ef4444',size=9,symbol='circle'),
        name='Wrist',showlegend=True))
    # Wrist path until the displayed frame, fixed image axes
    if p is not None and COORDINATE_MODE=='image' and SHOW_WRIST_TRAJECTORY:
        current=next((f for f in frames if coords[f] is p),None)
        trail=[coords[k][0] for k in frames if current is not None and k<=current and coords[k] is not None]
    else:
        trail=[]
    result.append(go.Scatter3d(
        x=[float(q[0]) for q in trail],y=[float(q[1]) for q in trail],
        z=[float(q[2]) for q in trail],mode='lines',
        line=dict(color='#fb7185',width=5),name='Wrist trajectory',showlegend=True))
    if COORDINATE_MODE=='image':
        result.append(go.Scatter3d(x=[0,1,1,0,0],y=[0,0,HEIGHT/WIDTH,HEIGHT/WIDTH,0],
            z=[0,0,0,0,0],mode='lines',line=dict(color='#64748b',width=3),
            name='Image boundary',showlegend=True))
    return result
fig=go.Figure(data=traces(coords[START_FRAME]),
              frames=[go.Frame(name=str(f),data=traces(coords[f]),
                     layout=go.Layout(title=f'Frame {f}: '+('Valid' if coords[f] is not None else 'Unavailable')))
                      for f in frames])
fig.update_layout(template='plotly_dark',height=730,
  title='FingerBall — image-origin hand skeleton (XY image, Z relative)',
  scene=dict(xaxis=dict(title='Image X / width',range=X_RANGE),yaxis=dict(title='Image Y / width (down positive in image mode)',range=Y_RANGE),
     zaxis=dict(title='Wrist-relative Z (not metric)',range=Z_RANGE),
     aspectmode='manual',aspectratio=ASPECT,camera=dict(eye=dict(x=1.6,y=1.2,z=1.2)),dragmode='orbit'),
  uirevision='preserve_camera',
  sliders=[dict(currentvalue=dict(prefix='Frame '),steps=[
    dict(label=str(f),method='animate',args=[[str(f)],
       dict(mode='immediate',frame=dict(duration=0,redraw=True),transition=dict(duration=0))])
    for f in frames])],
  updatemenus=[dict(type='buttons',showactive=False,buttons=[
      dict(label='Play',method='animate',args=[None,dict(frame=dict(duration=300,redraw=True),fromcurrent=True)]),
      dict(label='Pause',method='animate',args=[[None],dict(mode='immediate',frame=dict(duration=0,redraw=True))])])])
fig.show(config=dict(scrollZoom=True,displaylogo=False))

## 4. 画像座標原点を維持したインタラクティブHTMLを保存

HTML上の時間スライダーで手首の画像内移動を確認し、Yaw/Pitchで360°回転できます。画像枠はZ=0の平面上に表示します。

In [ ]:
html_path=out_dir/'FingerBall_Stage2F_interactive_3D.html'
html=pio.to_html(fig,include_plotlyjs=True,full_html=True,div_id='fingerball3d')
controls="""<div style="padding:16px;background:#111827;color:white;font-family:sans-serif">
横回転 Yaw <input id="fbYaw" type="range" min="-180" max="180" value="40"><span id="yawVal">40°</span>
縦回転 Pitch <input id="fbPitch" type="range" min="-180" max="180" value="25"><span id="pitchVal">25°</span>
<button id="fbFront">正面</button><button id="fbSide">側面</button><button id="fbTop">上面</button>
<div style="font-size:12px">画像左上原点・XYは画像位置、Zは手首相対。実空間のカメラ座標系ではありません。</div></div>"""
script="""<script>
(function(){
const gd=document.getElementById('fingerball3d'),
y=document.getElementById('fbYaw'),p=document.getElementById('fbPitch');
function redraw(){
const a=Number(y.value)*Math.PI/180,b=Number(p.value)*Math.PI/180,r=2.4;
Plotly.relayout(gd,{'scene.camera.eye':{x:r*Math.cos(b)*Math.cos(a),y:r*Math.cos(b)*Math.sin(a),z:r*Math.sin(b)}});
document.getElementById('yawVal').textContent=y.value+'°';
document.getElementById('pitchVal').textContent=p.value+'°';
}
y.addEventListener('input',redraw);p.addEventListener('input',redraw);
function set(a,b){y.value=a;p.value=b;redraw();}
document.getElementById('fbFront').onclick=()=>set(0,0);
document.getElementById('fbSide').onclick=()=>set(90,0);
document.getElementById('fbTop').onclick=()=>set(0,89);
})();
</script>"""
html=html.replace('<body>','<body>'+controls,1).replace('</body>',script+'</body>',1)
html_path.write_text(html,encoding='utf-8')
print('HTML:',html_path,round(html_path.stat().st_size/1e6,2),'MB')

## 5. 代表フレームのYaw/Pitch各360° MP4と0/90/180/270°画像

In [ ]:
def render_view(p,az,elev):
    f=plt.figure(figsize=(5.7,5.7),dpi=95,facecolor='#101827')
    ax=f.add_subplot(111,projection='3d')
    ax.set_facecolor('#101827')
    for finger,edges in FINGERS.items():
        for a,b in edges:
            v=p[[a,b]]
            ax.plot(v[:,0],v[:,1],v[:,2],color=COLORS[finger],linewidth=3.2)
    for finger,j in TIPS.items():
        v=p[j];ax.scatter(*v,s=60,color=COLORS[finger],depthshade=False)
    ax.scatter(*p[0],s=90,color='#ef4444',depthshade=False)
    if COORDINATE_MODE=='image':
        xx=[0,1,1,0,0];yy=[0,0,HEIGHT/WIDTH,HEIGHT/WIDTH,0]
        ax.plot(xx,yy,[0]*5,color='#64748b',linewidth=1.5)
        ax.set_box_aspect((1,HEIGHT/WIDTH,.5))
    else:ax.set_box_aspect((1,1,1))
    ax.set(xlim=X_RANGE,ylim=Y_RANGE,zlim=Z_RANGE,
           xlabel='image X/width',ylabel='image Y/width',zlabel='relative Z')
    ax.view_init(elev=elev,azim=az)
    ax.tick_params(labelsize=7,colors='white')
    for label in (ax.xaxis.label,ax.yaxis.label,ax.zaxis.label):label.set_color('white')
    f.suptitle(f'Frame {REPRESENTATIVE_FRAME} | AI-estimated pseudo-3D',color='white')
    f.canvas.draw()
    rgba=np.asarray(f.canvas.buffer_rgba())
    bgr=cv2.cvtColor(rgba[:,:,:3].copy(),cv2.COLOR_RGB2BGR)
    plt.close(f)
    return bgr
reference=coords[REPRESENTATIVE_FRAME]
preview=render_view(reference,30,15)
hh,ww=preview.shape[:2]
def make_rotation(path,kind):
    writer=cv2.VideoWriter(str(path),cv2.VideoWriter_fourcc(*'mp4v'),ROTATION_FPS,(ww,hh))
    if not writer.isOpened():raise RuntimeError('MP4 writer failed')
    try:
        for i in tqdm(range(ROTATION_FRAMES),desc=kind):
            degrees=i*360/ROTATION_FRAMES
            az,elev=(degrees,15) if kind=='yaw' else (30,degrees)
            bgr=render_view(reference,az,elev)
            cv2.putText(bgr,f'{kind.upper()} {degrees:.0f} deg',(16,hh-18),
                cv2.FONT_HERSHEY_SIMPLEX,.65,(255,255,255),2)
            writer.write(bgr)
    finally:writer.release()
yaw_mp4=out_dir/f'frame_{REPRESENTATIVE_FRAME}_yaw_360.mp4'
pitch_mp4=out_dir/f'frame_{REPRESENTATIVE_FRAME}_pitch_360.mp4'
if EXPORT_MP4:
    make_rotation(yaw_mp4,'yaw')
    make_rotation(pitch_mp4,'pitch')
png_paths=[]
for kind in ['yaw','pitch']:
    for deg in [0,90,180,270]:
        az,el=(deg,15) if kind=='yaw' else (30,deg)
        p=out_dir/f'frame_{REPRESENTATIVE_FRAME}_{kind}_{deg:03d}.png'
        cv2.imwrite(str(p),render_view(reference,az,el))
        png_paths.append(p)
print('MP4:',yaw_mp4,pitch_mp4)

## 6. 共有用ZIPと解析条件を保存

In [ ]:
report={'source_csv':str(csv_path),'frames':[START_FRAME,END_FRAME],
        'valid_frames':valid,'representative_frame':REPRESENTATIVE_FRAME,
        'image_width_px':WIDTH,'image_height_px':HEIGHT,
        'coordinate_mode':COORDINATE_MODE,
        'XY_origin':'image top-left (0,0)' if COORDINATE_MODE=='image' else 'wrist',
        'Z_origin':'wrist-relative; absolute camera depth unavailable','relative_depth_multiplier':Z_DEPTH_MULTIPLIER,
        'limitations':['Single-view pseudo-3D, not metric 3D',
         'Depth is MediaPipe relative Z; no ground truth for unseen surface',
         'No validated ball release or joint angles',
         'Missing frames not imputed',
         'Image origin is not calibrated camera 3D space']}
report_file=out_dir/'stage2f_report.json'
report_file.write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
readme=out_dir/'README_for_trainer.txt'
readme.write_text('FingerBall Stage 2F: 手指骨格疑似3Dビューア\n'
  'HTMLの時間スライダーとYaw/Pitchスライダーで操作できます。\n'
  'MP4は代表フレームの横360度／縦360度回転表示です。\n'
  '奥行きは単眼MediaPipeの相対Zであり、実空間で計測した3Dではありません。\n'
  '共有前に個人情報・研究倫理・データ提供者の許諾をご確認ください。\n',encoding='utf-8')
package=out_dir/'FingerBall_Stage2F_3D_hand_viewer_package.zip'
assets=[html_path,report_file,readme]+png_paths
if EXPORT_MP4:assets += [yaw_mp4,pitch_mp4]
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as z:
    for item in assets:
        if not item.exists():raise FileNotFoundError(item)
        z.write(item,arcname=item.name)
print('ZIP:',package,'MB:',round(package.stat().st_size/1e6,2))
print('Results:',out_dir)

## 利用手順

1. Google Driveへ接続してStage 2D/2Eの21点CSVを読み込み。
2. Notebookの3D表示でフレームを選び、マウスで360度回転。
3. 出力HTMLを開けば **時間スライダー＋Yaw/Pitch回転バー**が使えます。
4. 代表フレームのYaw/Pitch回転MP4とZIPを確認してから共有。